# NCES Public School Data Download

### Importing necessary libraries

In [2]:
import time, os
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Setting up state mappings, columns to note, and chrome options for Selenium driver

In [3]:
# FIPS code for each State and Territory to use in URL construction for downloading school data for each state
state_FIPS = {
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'District of Columbia': '11',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56',
    'American Samoa': '60',
    'Guam': '66',
    'Northern Mariana Islands': '69',
    'Puerto Rico': '72',
    'U.S. Virgin Islands': '78'
}

keep_cols = ['NCES School ID', 'Low Grade', 'High Grade', 'School Name', 'District', 'County Name', 'Street Address', 'City',
             'State', 'ZIP', 'Phone', 'Locale Code', 'Charter', 'Students', 'Teachers', 'Free Lunch', 'Reduced Lunch', 'Type', 'Status']

final_cols = ['NCES School ID', 'Low Grade', 'High Grade', 'Account Name', 'School District', 'County Name', 'Billing Street',
              'Billing City', 'Billing State', 'Billing ZIP', 'Phone', 'School Environment', 'School Type',
              'Number of Students Served', 'Number of Teachers', 'Free Lunch', 'Reduced Lunch', 'Type', 'NCES Status']

# Set download directory
download_dir = 'C:\\Users\\mridd\\Desktop\\web_scraping_testing\\sch_data_download_testing\\public_sd_downloads'

# Configure Chrome options for automatic download
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir} # Establish and add download directory preference
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--headless=new")  # Run Chrome in headless mode (optional)

### Function to Standardize Columns Kept

In [8]:
def create_sd_csv(excel_file, state):
    df = pd.concat(excel_file, ignore_index=True)

    # Change column names to values in keep_cols
    df.columns = df.loc[5]
    df = df[keep_cols]
    df = df.loc[6:].reset_index(drop=True) # Get only rows containing school data, reset index

    # Create masks to handle empty values for different columns
    free_lunch_empty_mask = (df['Free Lunch'] == '–') | (df['Free Lunch'] == '†')
    reduced_lunch_empty_mask = (df['Reduced Lunch'] == '–') | (df['Reduced Lunch'] == '†')
    num_students_empty_mask = (df['Students'] == '–') | (df['Students'] == '†')
    num_teachers_empty_mask = (df['Teachers'] == '–') | (df['Teachers'] == '†')
    ungraded_grade_empty_mask = (df['Low Grade'] == '–') & (df['High Grade'] == '–')

    # Assign empty string to empty values, convert non-empty values to appropriate data types
    df.loc[free_lunch_empty_mask, 'Free Lunch'] = ''
    df.loc[~free_lunch_empty_mask, 'Free Lunch'] = df.loc[~free_lunch_empty_mask, 'Free Lunch'].astype(float).astype(int)

    df.loc[reduced_lunch_empty_mask, 'Reduced Lunch'] = ''
    df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'] = df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'].astype(float).astype(int)

    df.loc[num_students_empty_mask, 'Students'] = ''
    df.loc[~num_students_empty_mask, 'Students'] = df.loc[~num_students_empty_mask, 'Students'].astype(float).astype(int)

    df.loc[num_teachers_empty_mask, 'Teachers'] = ''
    df.loc[~num_teachers_empty_mask, 'Teachers'] = df.loc[~num_teachers_empty_mask, 'Teachers'].astype(float)

    # Change all values in column Type to 'School'
    df['Type'] = 'School'

    # Create Locale mappings/masks and update Locale column based on Locale Code values
    urban = [11, 12, 13]
    urban_mask = df['Locale Code'].astype(int).isin(urban)

    suburban = [21, 22, 23, 31, 32, 33]
    suburban_mask = df['Locale Code'].astype(int).isin(suburban)

    rural = [41, 42, 43]
    rural_mask = df['Locale Code'].astype(int).isin(rural)

    df.loc[urban_mask, 'Locale Code'] = 'Urban'
    df.loc[suburban_mask, 'Locale Code'] = 'Suburban'
    df.loc[rural_mask, 'Locale Code'] = 'Rural'

    # Remove leading zeros from Low Grade and High Grade
    values = ['01','02','03','04','05','06','07','08','09']
    df.loc[df['Low Grade'].isin(values), 'Low Grade'] = df['Low Grade'].str.lstrip('0')
    df.loc[df['High Grade'].isin(values), 'High Grade'] = df['High Grade'].str.lstrip('0')

    df.loc[ungraded_grade_empty_mask, ['Low Grade', 'High Grade']] = '' # Assign empty string to ungraded schools

    df.loc[df['High Grade'] == '12', 'High Grade'] = '12' # Change any High Grade values above 12 to 12

    # Change 'Status' values of 'Added', 'Changed Agency', and 'Reopened' to 'Open'
    change_status_to_open_mask = df['Status'].isin(['Added', 'Changed Agency', 'Reopened'])
    df.loc[change_status_to_open_mask, 'Status'] = 'Open'

    # Determine school types based on Charter status
    charter_mask = (df['Charter'] == 'Yes')

    df.loc[charter_mask, 'Charter'] = 'Public- Charter'
    df.loc[~charter_mask, 'Charter'] = 'Public'

    # Change state abbreviation to full name
    df['State'] = state

    # Update columns to final version we want
    df.columns = final_cols

    # Temp output
    df.to_csv(f'{download_dir}\\{state}_sd.csv', index=False)

### Main Loop to Retrieve, Standardize, and Save Files

In [ ]:
# Logic to navigate to School Data Excel file and download it
for state, fips in state_FIPS.items():
    # # Create Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/ccd/schoolsearch/school_list.asp?Search=1&InstName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&DistrictName=&DistrictID=&SchoolType=1&SchoolType=2&SchoolType=3&SchoolType=4&SpecificSchlTypes=all&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading public school data for {state}...')
    try:
        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        # Click on link that opens window with Excel download link
        wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass'))).click()
        time.sleep(20)

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window
        time.sleep(10) # Pause to allow time for download to occur

        # Rename the downloaded Excel file to test_download.csv
        files = os.listdir(download_dir)
        excel_files = [f for f in files if f.endswith('.xlsx') or f.endswith('.xls')]
        if excel_files:
            excel_file = pd.read_html(os.path.join(download_dir, excel_files[0])) # Read the downloaded Excel file into a DataFrame

            print(f'{state}: {excel_files[0]}')
            create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file

            os.remove(os.path.join(download_dir, excel_files[0])) # Remove the original Excel file after conversion to CSV

    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit()

### Testing Cell

In [40]:
excel_file = pd.read_html('ncesdata_2309FCC0.xls')
df = pd.concat(excel_file, ignore_index=True)

# Change column names to values in keep_cols
df.columns = df.loc[5]
df = df[keep_cols]
df = df.loc[6:].reset_index(drop=True) # Get only rows containing school data, reset index

# Create masks to handle empty values for different columns
free_lunch_empty_mask = (df['Free Lunch'] == '–') | (df['Free Lunch'] == '†')
reduced_lunch_empty_mask = (df['Reduced Lunch'] == '–') | (df['Reduced Lunch'] == '†')
num_students_empty_mask = (df['Students'] == '–') | (df['Students'] == '†')
num_teachers_empty_mask = (df['Teachers'] == '–') | (df['Teachers'] == '†')
ungraded_grade_empty_mask = (df['Low Grade'] == '–') & (df['High Grade'] == '–')

# Assign empty string to empty values, convert non-empty values to appropriate data types
df.loc[free_lunch_empty_mask, 'Free Lunch'] = ''
df.loc[~free_lunch_empty_mask, 'Free Lunch'] = df.loc[~free_lunch_empty_mask, 'Free Lunch'].astype(float).astype(int)

df.loc[reduced_lunch_empty_mask, 'Reduced Lunch'] = ''
df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'] = df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'].astype(float).astype(int)

df.loc[num_students_empty_mask, 'Students'] = ''
df.loc[~num_students_empty_mask, 'Students'] = df.loc[~num_students_empty_mask, 'Students'].astype(float).astype(int)

df.loc[num_teachers_empty_mask, 'Teachers'] = ''
df.loc[~num_teachers_empty_mask, 'Teachers'] = df.loc[~num_teachers_empty_mask, 'Teachers'].astype(float)

# Change all values in column Type to 'School'
df['Type'] = 'School'

# Create Locale mappings/masks and update Locale column based on Locale Code values
urban = [11, 12, 13]
urban_mask = df['Locale Code'].astype(int).isin(urban)

suburban = [21, 22, 23, 31, 32, 33]
suburban_mask = df['Locale Code'].astype(int).isin(suburban)

rural = [41, 42, 43]
rural_mask = df['Locale Code'].astype(int).isin(rural)

df.loc[urban_mask, 'Locale Code'] = 'Urban'
df.loc[suburban_mask, 'Locale Code'] = 'Suburban'
df.loc[rural_mask, 'Locale Code'] = 'Rural'

# Remove leading zeros from Low Grade and High Grade
values = ['01','02','03','04','05','06','07','08','09']
df.loc[df['Low Grade'].isin(values), 'Low Grade'] = df['Low Grade'].str.lstrip('0')
df.loc[df['High Grade'].isin(values), 'High Grade'] = df['High Grade'].str.lstrip('0')

df.loc[ungraded_grade_empty_mask, ['Low Grade', 'High Grade']] = '' # Assign empty string to ungraded schools

# Determine school types based on Charter status
charter_mask = (df['Charter'] == 'Yes')

df.loc[charter_mask, 'Charter'] = 'Public- Charter'
df.loc[~charter_mask, 'Charter'] = 'Public'

# Temp output
df.to_csv('test_download.csv', index=False)

